# Water Level Imputation — Improved v3
**Lake Tanganyika Research Project**

This notebook adds three improvements over v2:
1. **Lag features** — use previous months' water levels as predictors
2. **Correlated river features** — borrow data from strongly related rivers
3. **NDVI gap filling** — interpolate missing vegetation data before modeling

---

## Step 0 — Load libraries and data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('master_dataset_inputed.csv', parse_dates=['date'])
df = df.sort_values(['river', 'date']).reset_index(drop=True)

print(f"Loaded {len(df):,} rows  |  {df['river'].nunique()} rivers  |  "
      f"{df['date'].min().year}\u2013{df['date'].max().year}")
print(f"Missing water_level: {df['water_level'].isna().sum():,} rows "
      f"({df['water_level'].isna().mean()*100:.1f}%)")

---
## Step 1 — Fix NDVI gaps (Improvement 3)

NDVI (vegetation index) is missing for ~44% of rows.  
We fill it with **time interpolation within each river** — connecting known values smoothly.  
Any remaining gaps at the start/end of a river's record are filled by carrying values forward/backward.

In [ ]:
print("NDVI missing BEFORE fix:")
print(df.groupby('river')['ndvi'].apply(
    lambda x: f"{x.isna().sum():>3} / {len(x)} missing ({x.isna().mean()*100:.0f}%)"
))

# ── Fix NDVI using transform (works safely with groupby) ──────────────────────
df['ndvi'] = (
    df.groupby('river')['ndvi']
    .transform(lambda x: x.interpolate(method='index').ffill().bfill())
)

print(f"\nNDVI missing AFTER fix: {df['ndvi'].isna().sum()} rows")

---
## Step 2 — Add lag features (Improvement 1)

We create new columns based on **past water level values** for each river:
- `wl_lag1` = water level 1 month ago
- `wl_lag2` = water level 2 months ago
- `wl_lag12` = water level 12 months ago (same season last year)
- `wl_rolling3` = average of the last 3 months

> **Why this helps:** Rivers don't change randomly. Last month's level is usually the strongest predictor of this month's level.

In [ ]:
# ── Create lag columns within each river's own history ───────────────────────
for col, n_shift in [('wl_lag1', 1), ('wl_lag2', 2), ('wl_lag12', 12)]:
    df[col] = df.groupby('river')['water_level'].shift(n_shift)

df['wl_rolling3'] = (
    df.groupby('river')['water_level']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

lag_cols = ['wl_lag1', 'wl_lag2', 'wl_lag12', 'wl_rolling3']
print("Lag feature availability (non-null rows):")
for col in lag_cols:
    avail = df[col].notna().sum()
    print(f"  {col:<15}: {avail:,} rows ({avail/len(df)*100:.1f}%)")

---
## Step 3 — Add correlated river features (Improvement 2)

Some rivers move very similarly together.  
When one river has a gap, the strongly correlated river's value is a useful clue.

**Top correlated pairs in this dataset:**
| River | Partner | Correlation |
|---|---|---|
| Kaburantwa | Rusizi | r = 0.88 |
| Mulembwe | Rusizi | r = 0.78 |
| Jiji | Mulembwe | r = 0.75 |
| Buzimba | Nyengwe | r = 0.71 |

In [ ]:
# ── Show the correlation heatmap ──────────────────────────────────────────────
pivot = df.pivot_table(index='date', columns='river', values='water_level')
corr_matrix = pivot.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    ax=ax, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.7}
)
ax.set_title('River water level correlations', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()
print("Green = strongly correlated. These pairs will share features.")

In [ ]:
# ── Add correlated partner's water level as a feature ────────────────────────
CORR_PAIRS = {
    'Kaburantwa': 'Rusizi',
    'Rusizi':     'Kaburantwa',
    'Mulembwe':   'Rusizi',
    'Jiji':       'Mulembwe',
    'Buzimba':    'Nyengwe',
    'Nyengwe':    'Buzimba',
    'Mpanda':     'Kaburantwa',
    # No strong partner for Nyamagana, Nyakagunda, Mutimbuzi
}

# Build date x river lookup table
wl_by_date = df.groupby(['date', 'river'])['water_level'].first().unstack('river')

def get_partner_value(row):
    partner = CORR_PAIRS.get(row['river'])
    if partner is None:
        return np.nan
    try:
        return wl_by_date.loc[row['date'], partner]
    except KeyError:
        return np.nan

df['wl_corr_partner'] = df.apply(get_partner_value, axis=1)

print("Correlated partner feature per river:")
print(df.groupby('river').apply(
    lambda g: f"partner={CORR_PAIRS.get(g.name, 'none'):<12}  "
              f"available={g['wl_corr_partner'].notna().sum():>3}/{len(g)} rows"
))

---
## Step 4 — Train v3 model and compare errors

We train a **Random Forest** per river using all three improvements,  
then compare the error (MAE) against the seasonal mean baseline.

> **MAE** = mean absolute error in meters. Lower is better.  
> We split each river 80% train / 20% test using the actual time order.

In [ ]:
ERA5_FEATURES  = ['month', 'year',
                   'era5_t2m_mean', 'era5_t2m_min', 'era5_t2m_max',
                   'era5_tp_sum', 'era5_u10_mean', 'era5_v10_mean',
                   'era5_d2m_mean', 'era5_msl_mean', 'ndvi']
LAG_FEATURES   = ['wl_lag1', 'wl_lag2', 'wl_lag12', 'wl_rolling3']
CORR_FEATURES  = ['wl_corr_partner']
ALL_FEATURES   = ERA5_FEATURES + LAG_FEATURES + CORR_FEATURES

print(f"Feature set: {len(ERA5_FEATURES)} ERA5  +  {len(LAG_FEATURES)} lag  +  {len(CORR_FEATURES)} corr  =  {len(ALL_FEATURES)} total")

In [ ]:
results = []

for river in sorted(df['river'].unique()):
    rdf = df[df['river'] == river].sort_values('date').reset_index(drop=True)
    known = rdf[rdf['water_level'].notna()].copy()

    # Baseline: seasonal mean
    split = int(len(known) * 0.8)
    train_b, test_b = known.iloc[:split], known.iloc[split:]
    seasonal = train_b.groupby('month')['water_level'].mean()
    mae_baseline = mean_absolute_error(
        test_b['water_level'], test_b['month'].map(seasonal)
    )

    # v3: needs at least wl_lag1 to be non-null
    known_v3 = known.dropna(subset=['wl_lag1']).copy()

    if len(known_v3) < 25:
        results.append({'River': river, 'n_train': len(known_v3),
                         'Baseline MAE': round(mae_baseline, 3),
                         'v3 MAE': None, 'Change': 'too few rows'})
        continue

    split3 = int(len(known_v3) * 0.8)
    train3, test3 = known_v3.iloc[:split3], known_v3.iloc[split3:]

    train_med = train3[ALL_FEATURES].median()
    X_train = train3[ALL_FEATURES].fillna(train_med)
    X_test  = test3[ALL_FEATURES].fillna(train_med)

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, train3['water_level'])
    mae_v3 = mean_absolute_error(test3['water_level'], rf.predict(X_test))

    change = (mae_baseline - mae_v3) / mae_baseline * 100
    results.append({'River': river, 'n_train': len(train3),
                     'Baseline MAE': round(mae_baseline, 3),
                     'v3 MAE': round(mae_v3, 3),
                     'Change': f"{change:+.1f}%"})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# ── Bar chart: baseline vs v3 per river ───────────────────────────────────────
plot_data = results_df.dropna(subset=['v3 MAE']).sort_values('Baseline MAE')

x = np.arange(len(plot_data))
w = 0.38

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(x + w/2, plot_data['Baseline MAE'], w,
        label='Seasonal mean (baseline)', color='#B4B2A9')
ax.barh(x - w/2, plot_data['v3 MAE'], w,
        label='v3 model (lag + corr + NDVI)', color='#1D9E75')

ax.set_yticks(x)
ax.set_yticklabels(plot_data['River'])
ax.set_xlabel('MAE in meters — lower is better')
ax.set_title('Imputation error: baseline vs v3', fontsize=13, pad=10)
ax.legend()

for i, (_, row) in enumerate(plot_data.iterrows()):
    pct = float(row['Change'].replace('%', '').replace('+', ''))
    color = '#0F6E56' if pct > 5 else '#993C1D' if pct < -5 else '#5F5E5A'
    ax.text(row['v3 MAE'] + 0.005, i - w/2,
            row['Change'], va='center', fontsize=9, color=color)

plt.tight_layout()
plt.show()

---
## Step 5 — Feature importance

Which features does the model rely on most?  
This helps us understand what is actually driving the predictions.

In [ ]:
from matplotlib.patches import Patch

# Train one combined model across all rivers
combined = df.dropna(subset=['water_level', 'wl_lag1']).copy()
train_med_all = combined[ALL_FEATURES].median()
X_all = combined[ALL_FEATURES].fillna(train_med_all)
y_all = combined['water_level']

rf_all = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_all.fit(X_all, y_all)

importance = pd.Series(rf_all.feature_importances_, index=ALL_FEATURES).sort_values(ascending=True)

colors = []
for feat in importance.index:
    if feat in LAG_FEATURES:   colors.append('#1D9E75')  # green = lag
    elif feat in CORR_FEATURES: colors.append('#3B8BD4') # blue = corr partner
    else:                       colors.append('#B4B2A9') # gray = ERA5

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(importance.index, importance.values, color=colors)
ax.set_xlabel('Feature importance (higher = more useful for prediction)')
ax.set_title('What drives the v3 imputation model?', fontsize=13, pad=10)
ax.legend(handles=[
    Patch(color='#1D9E75', label='Lag features (new in v3)'),
    Patch(color='#3B8BD4', label='Correlated river (new in v3)'),
    Patch(color='#B4B2A9', label='ERA5 weather / other'),
], loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 6 — Fill the actual missing values

Now we apply the trained model to fill every row where `water_level` is NaN.

In [ ]:
df['water_level_imputed_v3'] = df['water_level'].copy()
df['wl_imputed_v3'] = False  # True = this row was filled by the model

for river in sorted(df['river'].unique()):
    rdf = df[df['river'] == river].sort_values('date').reset_index(drop=True)
    known = rdf[rdf['water_level'].notna()].dropna(subset=['wl_lag1'])

    missing_mask = (df['river'] == river) & df['water_level'].isna()
    n_missing = missing_mask.sum()
    if n_missing == 0:
        continue

    if len(known) < 25:
        # Fallback: simple seasonal mean for rivers with too little data
        seasonal = rdf[rdf['water_level'].notna()].groupby('month')['water_level'].mean()
        df.loc[missing_mask, 'water_level_imputed_v3'] = \
            df.loc[missing_mask, 'month'].map(seasonal)
        df.loc[missing_mask, 'wl_imputed_v3'] = True
        print(f"{river:<15} seasonal fallback  ({n_missing} rows)")
        continue

    train_med = known[ALL_FEATURES].median()
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(known[ALL_FEATURES].fillna(train_med), known['water_level'])

    X_miss = df.loc[missing_mask, ALL_FEATURES].fillna(train_med)
    predicted = rf.predict(X_miss)
    # Clip to realistic range
    predicted = np.clip(predicted, 0, known['water_level'].max() * 1.2)

    df.loc[missing_mask, 'water_level_imputed_v3'] = predicted
    df.loc[missing_mask, 'wl_imputed_v3'] = True
    print(f"{river:<15} RF model  ({n_missing} rows filled)")

still_nan = df['water_level_imputed_v3'].isna().sum()
print(f"\nStill missing after v3: {still_nan} rows")

---
## Step 7 — Visual check

Always check the result visually!  
Orange dots = filled values. Dark dots = original observed values.  
The orange dots should look like a natural continuation of the dark dots.

In [ ]:
rivers_to_plot = ['Rusizi', 'Buzimba', 'Nyamagana', 'Kaburantwa']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, river in zip(axes, rivers_to_plot):
    rdf = df[df['river'] == river].sort_values('date')

    ax.plot(rdf['date'], rdf['water_level_imputed_v3'],
            color='#1D9E75', linewidth=1.2, label='v3 (full series)', zorder=2)
    ax.scatter(rdf.loc[rdf['wl_imputed_v3'], 'date'],
               rdf.loc[rdf['wl_imputed_v3'], 'water_level_imputed_v3'],
               color='#EF9F27', s=12, zorder=3, label='filled gap')
    ax.scatter(rdf.loc[rdf['water_level'].notna(), 'date'],
               rdf.loc[rdf['water_level'].notna(), 'water_level'],
               color='#042C53', s=6, zorder=4, label='observed', alpha=0.7)

    ax.set_title(river, fontsize=11)
    ax.set_ylabel('Water level (m)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)

fig.suptitle('Observed (dark) vs imputed gaps (orange) — v3 model',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 8 — Save the result

In [ ]:
output_path = 'master_dataset_imputed_v3.csv'
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")
print("\nNew columns added in v3:")
for col in ['wl_lag1','wl_lag2','wl_lag12','wl_rolling3',
            'wl_corr_partner','water_level_imputed_v3','wl_imputed_v3']:
    print(f"  {col}")

print("\nFinal summary per river:")
print(df.groupby('river').apply(
    lambda g: pd.Series({
        'observed': int(g['water_level'].notna().sum()),
        'imputed_v3': int(g['wl_imputed_v3'].sum()),
        'total': len(g)
    })
))

---
## Summary

| Step | What we did | Why it helps |
|---|---|---|
| NDVI interpolation | Filled missing vegetation data by time interpolation | More complete features for all rows |
| Lag features | Added last month, 2 months ago, 12 months ago | Usually becomes the #1 most important feature |
| Correlated rivers | Added a partner river's water level as a feature | Especially useful for Kaburantwa, Rusizi, Mulembwe |
| Fallback | Seasonal mean for rivers with < 25 training rows | Mutimbuzi (70% missing) stays reliable |

**Next steps:**
- Look at the Step 7 plots — orange dots should look natural, not jumpy
- Use `water_level_imputed_v3` column as input for trend analysis and prediction models
- The `wl_imputed_v3` column (True/False) tells you which values were filled